In [1]:
import pandas as pd

df = pd.read_excel("../data/raw/6S_AI_TASK-Loan_default_Loan_default.xlsx")
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [6]:
id_col = 'LoanID'
target_col = 'Default'

num_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 
            'InterestRate', 'LoanTerm', 'DTIRatio']
cat_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 
            'LoanPurpose', 'HasCoSigner']

In [7]:
all_cols = set(num_cols + cat_cols + [id_col, target_col])

missing = set(df.columns) - all_cols
extra = all_cols - set(df.columns)

print("Missing from schema:", missing)
print("Extra in schema:", extra)

Missing from schema: set()
Extra in schema: set()


In [ ]:
def engineer_feature(df: pd.DataFrame) -> pd.DataFrame:
    df['Loan_to_Income'] = df['LoanAmount'] / df['Income']
    df['Income_per_CreditLine'] = df['Income'] / (df['NumCreditLines'] + 1)  # Avoid division by zero
    df['Loan_per_CreditLine'] = df['LoanAmount'] / (df['NumCreditLines'] + 1)  # Avoid division by zero

    df['Risk_Interaction'] = df['CreditScore'] * df['InterestRate']
    df['Debt_Stress'] = df['DTIRatio'] * df['LoanAmount']

    df['Employment_Stability'] = df['MonthsEmployed'] / (df['Age'] - 18 + 1)  # Avoid division by zero

    df['CreditScore_bin'] = pd.cut(df['CreditScore'], bins=5)
    df['DTI_bin'] = pd.cut(df['DTIRatio'], bins=[0, 0.3, 0.6, 1])

    binary_map = {'Yes': 1, 'No': 0}

    df['HasMortgage'] = df['HasMortgage'].map(binary_map)
    df['HasDependents'] = df['HasDependents'].map(binary_map)   
    df['HasCoSigner'] = df['HasCoSigner'].map(binary_map)
    return df